In [1]:
# ============================================================
# 03 · Cell 1 — Setup and data load
# ============================================================
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

SRC_DIR = Path.cwd() / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from commodity_factors.config import PATHS, UNIVERSE, Sector

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
plt.style.use("seaborn-v0_8-whitegrid")

BACKTEST_START = "2002-01-31"

daily_returns = pd.read_parquet(PATHS.processed / "returns_daily.parquet")
monthly_returns = pd.read_parquet(PATHS.processed / "returns_monthly.parquet")
return_index = pd.read_parquet(PATHS.processed / "return_index.parquet")
prices = pd.read_parquet(PATHS.processed / "prices_clean.parquet")
curves = pd.read_parquet(PATHS.raw / "eia_curves.parquet")

for name, frame in {
    "daily_returns": daily_returns,
    "monthly_returns": monthly_returns,
    "return_index": return_index,
    "prices": prices,
    "curves": curves,
}.items():
    print(f"{name:<16}: {frame.shape[0]:>6,} rows x {frame.shape[1]:>2} cols | "
          f"{frame.index.min():%Y-%m-%d} -> {frame.index.max():%Y-%m-%d}")

daily_returns   :  6,754 rows x 26 cols | 2000-01-03 -> 2026-10-16
monthly_returns :    322 rows x 26 cols | 2000-01-31 -> 2026-10-31
return_index    :  6,754 rows x 26 cols | 2000-01-03 -> 2026-10-16
prices          :  6,754 rows x 26 cols | 2000-01-03 -> 2026-10-16
curves          : 11,127 rows x 15 cols | 1980-01-02 -> 2024-04-05


In [2]:
%%writefile src/commodity_factors/signals.py
"""Monthly factor signals for the commodity futures universe.

Every signal is a DataFrame indexed by month-end (rows) and contract (columns).
The value at month-end t uses only information available at the close of t;
portfolios formed on it earn returns in month t+1.
"""

from __future__ import annotations

from abc import ABC, abstractmethod
from typing import ClassVar

import numpy as np
import pandas as pd


def month_end_last(frame: pd.DataFrame) -> pd.DataFrame:
    """Last available observation in each calendar month, indexed by month-end."""
    return frame.resample("ME").last()


class Signal(ABC):
    """Base class for a monthly signal panel (month-end x contract)."""

    name: ClassVar[str]

    @abstractmethod
    def compute(self) -> pd.DataFrame:
        """Return the signal panel."""

    def summary(self) -> pd.DataFrame:
        """Per-contract coverage and distribution of the signal."""
        signal = self.compute()
        n_obs = signal.notna().sum()
        return pd.DataFrame(
            {
                "first_month": signal.apply(lambda s: s.first_valid_index()),
                "n_months": n_obs,
                "mean": signal.mean(),
                "std": signal.std(),
                "pct_positive": signal.gt(0).sum().div(n_obs).mul(100),
            }
        )


class MomentumSignal(Signal):
    """Cumulative excess return from month t-lookback to t-skip (cross-sectional momentum)."""

    name = "momentum"

    def __init__(self, return_index: pd.DataFrame, lookback: int = 12, skip: int = 1) -> None:
        if not 0 <= skip < lookback:
            raise ValueError("Require 0 <= skip < lookback.")
        self._return_index = return_index
        self._lookback = lookback
        self._skip = skip

    def compute(self) -> pd.DataFrame:
        monthly = month_end_last(self._return_index)
        return monthly.shift(self._skip) / monthly.shift(self._lookback) - 1


class TimeSeriesMomentumSignal(MomentumSignal):
    """Sign of each contract's own trailing excess return: +1 long, -1 short."""

    name = "tsmom"

    def __init__(self, return_index: pd.DataFrame, lookback: int = 12) -> None:
        super().__init__(return_index, lookback=lookback, skip=0)

    def compute(self) -> pd.DataFrame:
        return np.sign(super().compute())

Writing src/commodity_factors/signals.py


In [3]:
# ============================================================
# 03 · Cell 3 — Trim partial month and compute momentum signals
# ============================================================
from commodity_factors.signals import MomentumSignal, TimeSeriesMomentumSignal, month_end_last

# 1. Last complete month
last_day = daily_returns.index[-1]
is_month_close = last_day == last_day + pd.offsets.BMonthEnd(0)
LAST_COMPLETE_MONTH = last_day + pd.offsets.MonthEnd(0) if is_month_close else last_day - pd.offsets.MonthEnd(1)
monthly_returns = monthly_returns.loc[:LAST_COMPLETE_MONTH]
print(f"Last daily date: {last_day:%Y-%m-%d} | last complete month: {LAST_COMPLETE_MONTH:%Y-%m-%d}\n")

# 2. Signals
signal_models = {
    "mom_12_1": MomentumSignal(return_index, lookback=12, skip=1),
    "mom_12_0": MomentumSignal(return_index, lookback=12, skip=0),
    "tsmom_12": TimeSeriesMomentumSignal(return_index, lookback=12),
}
signals = {name: model.compute().loc[:LAST_COMPLETE_MONTH] for name, model in signal_models.items()}

# 3. Timing check (WTI, signal at end of Dec 2024)
month_index = month_end_last(return_index)["CL"]
manual_12_1 = month_index.loc["2024-11-30"] / month_index.loc["2023-12-31"] - 1
print("Timing check: CL momentum 12-1 at 2024-12-31")
print(f"  manual (Dec-2023 -> Nov-2024): {manual_12_1:.6f}")
print(f"  signal panel                 : {signals['mom_12_1'].at[pd.Timestamp('2024-12-31'), 'CL']:.6f}\n")

# 4. Coverage from backtest start
coverage = signals["mom_12_1"].loc[BACKTEST_START:].notna().sum(axis=1)
print("Contracts with a valid momentum signal (yearly average)")
print(coverage.groupby(coverage.index.year).mean().round(1).to_frame("n").T.to_string(), "\n")


# 5. Predictive preview
def rank_ic(signal: pd.DataFrame, forward: pd.DataFrame, min_assets: int = 10) -> pd.Series:
    """Monthly Spearman correlation between signal and next-month returns."""
    signal, forward = signal.align(forward, join="inner")
    valid = signal.notna() & forward.notna()
    ic = signal.where(valid).rank(axis=1).corrwith(forward.where(valid).rank(axis=1), axis=1)
    return ic[valid.sum(axis=1) >= min_assets]


forward_returns = monthly_returns.shift(-1)

ic_table = {}
for name in ["mom_12_1", "mom_12_0"]:
    ic = rank_ic(signals[name].loc[BACKTEST_START:], forward_returns).dropna()
    ic_table[name] = {
        "mean_ic": ic.mean(),
        "t_stat": ic.mean() / ic.std() * np.sqrt(len(ic)),
        "pct_positive": ic.gt(0).mean() * 100,
        "n_months": len(ic),
    }
print("Rank IC preview: signal at t vs return in t+1")
print(pd.DataFrame(ic_table).T.round(3).to_string(), "\n")

tsmom, fwd = signals["tsmom_12"].loc[BACKTEST_START:].align(forward_returns, join="inner")
long_ret = fwd.where(tsmom > 0).stack().mean() * 100
short_ret = fwd.where(tsmom < 0).stack().mean() * 100
print("TSMOM preview: average next-month return (%)")
print(f"  when long  : {long_ret:.3f}")
print(f"  when short : {short_ret:.3f}")
print(f"  share of positions long: {tsmom.gt(0).sum().sum() / tsmom.notna().sum().sum() * 100:.1f}%")

Last daily date: 2026-10-16 | last complete month: 2026-09-30

Timing check: CL momentum 12-1 at 2024-12-31
  manual (Dec-2023 -> Nov-2024): 0.094389
  signal panel                 : 0.094389

Contracts with a valid momentum signal (yearly average)
date  2002  2003  2004  2005  2006  2007  2008  2009  2010  2011  2012  2013  2014  2015  2016  2017  2018  2019  2020  2021  2022  2023  2024  2025  2026
n     20.1  21.0  21.0  22.0  22.6  23.0  23.5  24.0  24.0  26.0  26.0  26.0  26.0  26.0  26.0  26.0  26.0  26.0  26.0  26.0  26.0  26.0  26.0  26.0  26.0 

Rank IC preview: signal at t vs return in t+1
          mean_ic  t_stat  pct_positive  n_months
mom_12_1    0.015   0.947        50.338     296.0
mom_12_0    0.010   0.640        53.378     296.0 

TSMOM preview: average next-month return (%)
  when long  : 0.849
  when short : 0.719
  share of positions long: 55.3%


In [4]:
%%writefile -a src/commodity_factors/signals.py


def drop_incomplete_last_month(monthly: pd.DataFrame, last_date: pd.Timestamp) -> pd.DataFrame:
    """Drop the final month if the source data ends before that month's last business day."""
    if last_date == last_date + pd.offsets.BMonthEnd(0):
        return monthly
    return monthly.loc[: last_date - pd.offsets.MonthEnd(1)]


class CarrySignal(Signal):
    """Annualised front/next futures spread, optionally seasonally adjusted.

    Raw carry = ln(F1 / F2) x periods_per_year (12 for monthly-listed contracts);
    positive means backwardation. Seasonal adjustment subtracts the average raw
    carry observed in the same calendar month in prior years (expanding window,
    strictly past data), so calendar-driven curve shape is not mistaken for scarcity.
    """

    name = "carry"

    def __init__(
        self,
        curves: pd.DataFrame,
        periods_per_year: int = 12,
        seasonal_adjust: bool = True,
        min_years: int = 3,
    ) -> None:
        self._curves = curves
        self._periods_per_year = periods_per_year
        self._seasonal_adjust = seasonal_adjust
        self._min_years = min_years

    def raw(self) -> pd.DataFrame:
        """Month-end raw carry, before any seasonal adjustment."""
        symbols = self._curves.columns.get_level_values("symbol").unique()
        front = self._curves.xs(1, axis=1, level="contract")[symbols]
        second = self._curves.xs(2, axis=1, level="contract")[symbols]
        daily = np.log(front.where(front > 0) / second.where(second > 0)) * self._periods_per_year
        monthly = month_end_last(daily.dropna(how="all"))
        return drop_incomplete_last_month(monthly, self._curves.dropna(how="all").index.max())

    def seasonal_norm(self) -> pd.DataFrame:
        """Average raw carry in the same calendar month of prior years (strictly past)."""
        raw = self.raw()
        return raw.groupby(raw.index.month).transform(
            lambda x: x.expanding(min_periods=self._min_years).mean().shift(1)
        )

    def compute(self) -> pd.DataFrame:
        raw = self.raw()
        return raw - self.seasonal_norm() if self._seasonal_adjust else raw

Appending to src/commodity_factors/signals.py


In [5]:
# ============================================================
# 03 · Cell 5 — Carry: raw vs seasonally adjusted
# ============================================================
from commodity_factors.signals import CarrySignal

carry_raw = CarrySignal(curves, seasonal_adjust=False).compute()
carry_adj = CarrySignal(curves, seasonal_adjust=True).compute()
signals["carry_raw"], signals["carry_adj"] = carry_raw, carry_adj

print(f"Carry signal months : {carry_raw.index.min():%Y-%m} -> {carry_raw.index.max():%Y-%m}")
print("First adjusted month: "
      + ", ".join(f"{s} {carry_adj[s].first_valid_index():%Y-%m}" for s in carry_adj.columns), "\n")


# 1. Seasonality: dispersion of calendar-month averages (backtest period)
def seasonality_strength(frame: pd.DataFrame) -> pd.Series:
    sample = frame.loc[BACKTEST_START:]
    return sample.groupby(sample.index.month).mean().std()


print("Seasonality strength: std of the 12 calendar-month averages")
print(pd.DataFrame({"raw": seasonality_strength(carry_raw),
                    "adjusted": seasonality_strength(carry_adj)}).round(3).to_string(), "\n")

profile = pd.DataFrame(
    {f"{s}_{label}": frame[s].loc[BACKTEST_START:] for s in ["RB", "NG"]
     for label, frame in [("raw", carry_raw), ("adj", carry_adj)]}
)
print("Average carry by calendar month (RB and NG)")
print(profile.groupby(profile.index.month).mean().round(2).T.to_string(), "\n")


# 2. Time-series carry preview
def time_series_preview(signal: pd.DataFrame, forward: pd.DataFrame) -> pd.DataFrame:
    """Next-month returns by signal sign, and the return of a sign(signal) position."""
    signal, forward = signal.loc[BACKTEST_START:].align(forward, join="inner")
    rows = {}
    for symbol in signal.columns:
        valid = signal[symbol].notna() & forward[symbol].notna()
        x, y = signal[symbol][valid], forward[symbol][valid]
        position_ret = np.sign(x) * y
        rows[symbol] = {
            "months": len(x),
            "pct_long": x.gt(0).mean() * 100,
            "ret_long_%": y[x > 0].mean() * 100,
            "ret_short_%": y[x < 0].mean() * 100,
            "position_ret_%": position_ret.mean() * 100,
            "t_stat": position_ret.mean() / position_ret.std() * np.sqrt(len(position_ret)),
        }
    return pd.DataFrame(rows).T.round(2)


for label, frame in [("RAW carry", carry_raw), ("SEASONALLY ADJUSTED carry", carry_adj)]:
    print(f"Time-series preview: {label} (long if > 0, short if < 0)")
    print(time_series_preview(frame, forward_returns).to_string(), "\n")

Carry signal months : 1985-01 -> 2024-03
First adjusted month: CL 1988-01, HO 1997-02, NG 1997-01, RB 2008-10 

Seasonality strength: std of the 12 calendar-month averages
          raw  adjusted
symbol                 
CL      0.053     0.052
HO      0.089     0.123
NG      0.360     0.127
RB      0.400     0.121 

Average carry by calendar month (RB and NG)
date      1     2     3     4     5     6     7     8     9     10    11    12
RB_raw -0.14 -0.99  0.01  0.05  0.15  0.15  0.25  0.75  0.27  0.19  0.10 -0.04
RB_adj  0.08 -0.10 -0.02 -0.19 -0.04  0.04  0.11  0.19  0.12  0.24  0.11  0.06
NG_raw  0.05 -0.09 -0.37 -0.34 -0.19 -0.06 -0.20 -0.91 -0.99 -0.50 -0.00  0.14
NG_adj -0.33 -0.17 -0.20 -0.14  0.00  0.02 -0.02  0.14  0.02 -0.10 -0.10 -0.17 

Time-series preview: RAW carry (long if > 0, short if < 0)
    months  pct_long  ret_long_%  ret_short_%  position_ret_%  t_stat
CL   267.0     36.70        1.60         0.42            0.32    0.47
HO   267.0     41.95        1.35         0

In [6]:
# ============================================================
# 03 · Cell 6 — Roll-implied carry for all 26 contracts, validated against EIA
# ============================================================
from commodity_factors.config import ROLL_MASK_DAYS
from commodity_factors.contracts import CONTRACT_SPECS, RollCalendar, TradingCalendar

roll_calendar = RollCalendar(CONTRACT_SPECS, TradingCalendar(prices.index))


def roll_implied_carry(symbol: str) -> pd.Series:
    """Annualised carry implied by each roll gap, dated when the gap is observed."""
    series = prices[symbol].dropna()
    log_ret = np.log(series.where(series > 0)).diff()
    years = range(series.index[0].year, series.index[-1].year + 2)
    expiries = roll_calendar.last_trading_days(symbol, years)  # delivery month -> last trading day
    gap_months = [(nxt - cur).n for cur, nxt in zip(expiries.index[:-1], expiries.index[1:])]
    positions = series.index.searchsorted(pd.DatetimeIndex(expiries.iloc[:-1].to_numpy()), side="right")
    span = ROLL_MASK_DAYS.get(symbol, 1)

    values = {}
    for position, months in zip(positions, gap_months):
        if position == 0 or position + span > len(series):
            continue
        gap = log_ret.iloc[position : position + span].sum()
        values[series.index[position + span - 1]] = -gap * 12 / months
    return pd.Series(values, name=symbol, dtype=float)


roll_carry_events = {symbol: roll_implied_carry(symbol) for symbol in UNIVERSE.symbols}

# Month-end panel: hold the latest observed carry until the next roll (max 4 months)
roll_carry = (
    pd.DataFrame({s: ev.resample("ME").last() for s, ev in roll_carry_events.items()})
    .ffill(limit=4)
    .loc[:LAST_COMPLETE_MONTH]
)

# 1. Validation against true EIA carry (raw, month-end), 2002 -> Mar 2024
validation = {}
for symbol in ["CL", "HO", "NG", "RB"]:
    pair = pd.concat([roll_carry[symbol], carry_raw[symbol]], axis=1, keys=["roll_implied", "eia"])
    pair = pair.loc[BACKTEST_START:"2024-03-31"].dropna()
    validation[symbol] = {
        "months": len(pair),
        "correlation": pair.corr().iloc[0, 1],
        "sign_agreement_%": (np.sign(pair["roll_implied"]) == np.sign(pair["eia"])).mean() * 100,
        "mean_roll_implied": pair["roll_implied"].mean(),
        "mean_eia": pair["eia"].mean(),
    }
print("Validation: roll-implied carry vs EIA carry (2002 -> 2024-03)")
print(pd.DataFrame(validation).T.round(3).to_string(), "\n")

# 2. Coverage and level for all 26 contracts
overview = pd.DataFrame(
    {
        "sector": UNIVERSE.to_frame()["sector"],
        "n_rolls": {s: len(ev) for s, ev in roll_carry_events.items()},
        "mean_carry": roll_carry.loc[BACKTEST_START:].mean(),
        "median_carry": roll_carry.loc[BACKTEST_START:].median(),
        "pct_backwardated": roll_carry.loc[BACKTEST_START:].gt(0).mean() * 100,
        "valid_months_%": roll_carry.loc[BACKTEST_START:].notna().mean() * 100,
    }
)
print("Roll-implied carry by contract (annualised, backtest period)")
print(overview.round(3).to_string())

Validation: roll-implied carry vs EIA carry (2002 -> 2024-03)
    months  correlation  sign_agreement_%  mean_roll_implied  mean_eia
CL   267.0        0.591            65.169             -0.038    -0.057
HO   267.0        0.202            54.307              0.008    -0.003
NG   267.0        0.496            64.794             -0.167    -0.287
RB   222.0        0.205            63.964              0.081     0.057 

Roll-implied carry by contract (annualised, backtest period)
       sector  n_rolls  mean_carry  median_carry  pct_backwardated  valid_months_%
CL     Energy      313      -0.014        -0.011            47.475         100.000
BZ     Energy      231       0.032         0.051            42.088          77.441
NG     Energy      313      -0.172        -0.155            38.721         100.000
HO     Energy      313       0.017        -0.005            48.485         100.000
RB     Energy      311       0.086         0.078            59.259         100.000
GC     Metals      156

C:\Users\navon\AppData\Local\Temp\ipykernel_22140\4253940403.py:41: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  pair = pd.concat([roll_carry[symbol], carry_raw[symbol]], axis=1, keys=["roll_implied", "eia"])
C:\Users\navon\AppData\Local\Temp\ipykernel_22140\4253940403.py:41: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  pair = pd.concat([roll_carry[symbol], carry_raw[symbol]], axis=1, keys=["roll_implied", "eia"])
C:\Users\navon\AppData\Local\Temp\ipykernel_22140\4253940403.py:41: Pandas4W

In [7]:
# ============================================================
# 03 · Cell 7 — Trailing 12-month roll yield, validated against EIA
# ============================================================
WINDOW = 12


def trailing_roll_yield(symbol: str, months: int = WINDOW) -> pd.Series:
    """Minus the sum of log roll gaps over the trailing window (positive = backwardation)."""
    series = prices[symbol].dropna()
    log_ret = np.log(series.where(series > 0)).diff()
    positions = series.index.get_indexer(roll_calendar.roll_dates(symbol, series.index))
    span = ROLL_MASK_DAYS.get(symbol, 1)

    gaps = {
        series.index[p + span - 1]: log_ret.iloc[p : p + span].sum()
        for p in positions
        if p >= 1 and p + span <= len(series)
    }
    month_has_data = series.resample("ME").count().gt(0)
    monthly_gap = pd.Series(gaps, dtype=float).resample("ME").sum().reindex(month_has_data.index, fill_value=0.0)
    complete_window = month_has_data.astype(int).rolling(months).sum().eq(months)
    return (-monthly_gap.rolling(months).sum()).where(complete_window).rename(symbol)


roll_yield = pd.DataFrame({s: trailing_roll_yield(s) for s in UNIVERSE.symbols}).loc[:LAST_COMPLETE_MONTH]

# 1. Validation: EIA trailing 12-month average of daily raw carry
validation = {}
for symbol in ["CL", "HO", "NG", "RB"]:
    eia_daily = np.log(curves[(symbol, 1)] / curves[(symbol, 2)]) * 12
    eia_trailing = eia_daily.resample("ME").mean().rolling(WINDOW).mean()
    pair = pd.DataFrame({"roll_yield": roll_yield[symbol], "eia_12m": eia_trailing})
    pair = pair.loc[BACKTEST_START:"2024-03-31"].dropna()
    validation[symbol] = {
        "months": len(pair),
        "correlation": pair.corr().iloc[0, 1],
        "sign_agreement_%": (np.sign(pair["roll_yield"]) == np.sign(pair["eia_12m"])).mean() * 100,
        "mean_roll_yield": pair["roll_yield"].mean(),
        "mean_eia_12m": pair["eia_12m"].mean(),
    }
print("Validation: trailing 12m roll yield vs EIA trailing 12m carry (2002 -> 2024-03)")
print(pd.DataFrame(validation).T.round(3).to_string(), "\n")

# 2. Overview for all contracts
sample = roll_yield.loc[BACKTEST_START:]
overview = pd.DataFrame(
    {
        "sector": UNIVERSE.to_frame()["sector"],
        "first_month": roll_yield.apply(lambda s: s.first_valid_index().strftime("%Y-%m")),
        "mean": sample.mean(),
        "std": sample.std(),
        "pct_backwardated": sample.gt(0).mean() * 100,
    }
)
print("Trailing 12m roll yield by contract (backtest period)")
print(overview.round(3).to_string(), "\n")

# 3. Cross-sectional predictive preview (ranked across all available contracts)
ic = rank_ic(roll_yield.loc[BACKTEST_START:], forward_returns).dropna()
print("Rank IC preview: roll yield at t vs return in t+1")
print(f"  mean IC {ic.mean():.3f} | t-stat {ic.mean() / ic.std() * np.sqrt(len(ic)):.2f} | "
      f"positive {ic.gt(0).mean() * 100:.1f}% | months {len(ic)}")

Validation: trailing 12m roll yield vs EIA trailing 12m carry (2002 -> 2024-03)
    months  correlation  sign_agreement_%  mean_roll_yield  mean_eia_12m
CL   267.0        0.850            78.277           -0.041        -0.067
HO   267.0        0.781            77.903            0.003        -0.015
NG   267.0        0.827            83.146           -0.174        -0.289
RB   211.0        0.593            70.142            0.097         0.027 

Trailing 12m roll yield by contract (backtest period)
       sector first_month   mean    std  pct_backwardated
CL     Energy     2001-07 -0.020  0.171            44.108
BZ     Energy     2008-06  0.032  0.173            44.108
NG     Energy     2001-07 -0.194  0.331            27.946
HO     Energy     2001-08  0.014  0.184            51.852
RB     Energy     2001-10  0.081  0.165            70.370
GC     Metals     2001-07 -0.029  0.046            25.589
SI     Metals     2001-07 -0.019  0.075            41.077
HG     Metals     2001-07 -0.021  0

C:\Users\navon\anaconda3\envs\quant\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [8]:
%%writefile -a src/commodity_factors/signals.py


class RollYieldSignal(Signal):
    """Trailing realised roll yield: minus the sum of log roll gaps over the window.

    A stitched front-month series jumps by ln(new / old contract) on each roll date.
    Summing those gaps over 12 months measures the average slope of the curve the
    contract was rolled along (positive = backwardation). A full year of rolls covers
    every contract pair once, so seasonal curve shape cancels without adjustment.
    """

    name = "roll_yield"

    def __init__(
        self,
        prices: pd.DataFrame,
        roll_dates: dict[str, pd.DatetimeIndex],
        mask_days: dict[str, int] | None = None,
        window: int = 12,
    ) -> None:
        self._prices = prices
        self._roll_dates = roll_dates
        self._mask_days = mask_days or {}
        self._window = window

    def gaps(self, symbol: str) -> pd.Series:
        """Log roll gap observed at each roll, dated at the end of the masked window."""
        series = self._prices[symbol].dropna()
        log_ret = np.log(series.where(series > 0)).diff()
        positions = series.index.get_indexer(self._roll_dates.get(symbol, pd.DatetimeIndex([])))
        span = self._mask_days.get(symbol, 1)
        values = {
            series.index[p + span - 1]: log_ret.iloc[p : p + span].sum()
            for p in positions
            if p >= 1 and p + span <= len(series)
        }
        return pd.Series(values, dtype=float, name=symbol)

    def _trailing(self, symbol: str) -> pd.Series:
        series = self._prices[symbol].dropna()
        month_has_data = series.resample("ME").count().gt(0)
        monthly_gap = (
            self.gaps(symbol).resample("ME").sum().reindex(month_has_data.index, fill_value=0.0)
        )
        complete = month_has_data.astype(int).rolling(self._window).sum().eq(self._window)
        return (-monthly_gap.rolling(self._window).sum()).where(complete).rename(symbol)

    def compute(self) -> pd.DataFrame:
        return pd.DataFrame({symbol: self._trailing(symbol) for symbol in self._prices.columns})

Appending to src/commodity_factors/signals.py


In [9]:
# ============================================================
# 03 · Cell 9 — Final signals, consistency check, factor relationships, save
# ============================================================
from commodity_factors.signals import RollYieldSignal

# 1. Carry from the module, checked against the prototype
roll_dates = {s: roll_calendar.roll_dates(s, prices[s].dropna().index) for s in UNIVERSE.symbols}
carry = RollYieldSignal(prices, roll_dates, ROLL_MASK_DAYS).compute().loc[:LAST_COMPLETE_MONTH]

module_vals, proto_vals = carry.align(roll_yield, join="outer")
print("Module vs prototype")
print(f"  max abs difference : {(module_vals - proto_vals).abs().max().max():.2e}")
print(f"  same NaN pattern   : {module_vals.isna().equals(proto_vals.isna())}\n")

final_signals = {
    "mom_12_1": signals["mom_12_1"],
    "mom_12_0": signals["mom_12_0"],
    "tsmom_12": signals["tsmom_12"],
    "carry": carry,
    "carry_eia_raw": carry_raw,
    "carry_eia_adj": carry_adj,
}

# 2. Rank IC by subperiod
PERIODS = {
    "2002-2026": (BACKTEST_START, None),
    "2002-2008": (BACKTEST_START, "2008-12-31"),
    "2009-2026": ("2009-01-31", None),
}
ic_rows = {}
for name in ["mom_12_1", "mom_12_0", "carry"]:
    ic_all = rank_ic(final_signals[name].loc[BACKTEST_START:], forward_returns).dropna()
    for label, (start, end) in PERIODS.items():
        ic = ic_all.loc[start:end]
        ic_rows[(name, label)] = {
            "mean_ic": ic.mean(),
            "t_stat": ic.mean() / ic.std() * np.sqrt(len(ic)),
            "pct_positive": ic.gt(0).mean() * 100,
            "months": len(ic),
        }
print("Rank IC by subperiod (signal at t vs return in t+1)")
print(pd.DataFrame(ic_rows).T.round(3).to_string(), "\n")


# 3. Average cross-sectional rank correlation between signals
def mean_rank_corr(a: pd.DataFrame, b: pd.DataFrame, min_assets: int = 10) -> float:
    a, b = a.loc[BACKTEST_START:].align(b.loc[BACKTEST_START:], join="inner")
    valid = a.notna() & b.notna()
    corr = a.where(valid).rank(axis=1).corrwith(b.where(valid).rank(axis=1), axis=1)
    return corr[valid.sum(axis=1) >= min_assets].mean()


print("Average monthly rank correlation between signals")
for left, right in [("mom_12_1", "mom_12_0"), ("mom_12_1", "carry"), ("mom_12_0", "carry")]:
    print(f"  {left:<9} vs {right:<9}: {mean_rank_corr(final_signals[left], final_signals[right]):.3f}")

# 4. Save
signal_panel = pd.concat(final_signals, axis=1, names=["signal", "symbol"])
signal_panel.to_parquet(PATHS.processed / "signals.parquet")
print(f"\nSaved signals.parquet: {signal_panel.shape[0]} months x {signal_panel.shape[1]} columns "
      f"({len(final_signals)} signals)")

Module vs prototype
  max abs difference : 0.00e+00
  same NaN pattern   : True

Rank IC by subperiod (signal at t vs return in t+1)
                    mean_ic  t_stat  pct_positive  months
mom_12_1 2002-2026    0.015   0.947        50.338   296.0
         2002-2008    0.011   0.378        51.190    84.0
         2009-2026    0.016   0.879        50.000   212.0
mom_12_0 2002-2026    0.010   0.640        53.378   296.0
         2002-2008    0.016   0.553        50.000    84.0
         2009-2026    0.008   0.407        54.717   212.0
carry    2002-2026    0.032   2.215        55.068   296.0
         2002-2008    0.045   1.670        52.381    84.0
         2009-2026    0.028   1.574        56.132   212.0 

Average monthly rank correlation between signals
  mom_12_1  vs mom_12_0 : 0.936
  mom_12_1  vs carry    : 0.426
  mom_12_0  vs carry    : 0.424

Saved signals.parquet: 501 months x 112 columns (6 signals)


C:\Users\navon\AppData\Local\Temp\ipykernel_22140\3898787589.py:58: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  signal_panel = pd.concat(final_signals, axis=1, names=["signal", "symbol"])


In [10]:
%%writefile -a src/commodity_factors/signals.py


class ValueSignal(Signal):
    """Long-horizon price reversal: ln(average price 4.5-5.5 years ago / current price).

    Uses unadjusted front-month price levels (close to spot). Positive = cheap
    relative to its own history (Asness, Moskowitz & Pedersen, 2013).
    """

    name = "value"

    def __init__(
        self,
        prices: pd.DataFrame,
        lag_months: int = 54,
        window_months: int = 13,
        min_months: int = 10,
    ) -> None:
        self._prices = prices
        self._lag_months = lag_months
        self._window_months = window_months
        self._min_months = min_months

    def compute(self) -> pd.DataFrame:
        monthly = month_end_last(self._prices)
        monthly = monthly.where(monthly > 0)
        past_average = (
            monthly.shift(self._lag_months)
            .rolling(self._window_months, min_periods=self._min_months)
            .mean()
        )
        return np.log(past_average / monthly)


class SkewnessSignal(Signal):
    """Minus the skewness of daily excess returns over a trailing window.

    The factor is long low-skew and short high-skew (lottery-like) contracts
    (Fernandez-Perez, Frijns, Fuertes & Miffre, 2018), so the signal is -skew.
    """

    name = "skewness"

    def __init__(
        self, daily_returns: pd.DataFrame, window_days: int = 252, min_days: int = 200
    ) -> None:
        self._daily_returns = daily_returns
        self._window_days = window_days
        self._min_days = min_days

    def compute(self) -> pd.DataFrame:
        skew = self._daily_returns.rolling(self._window_days, min_periods=self._min_days).skew()
        return -month_end_last(skew)

Appending to src/commodity_factors/signals.py


In [11]:
# ============================================================
# 03 · Cell 11 — Value and skewness: compute, validate, preview
# ============================================================
from commodity_factors.signals import SkewnessSignal, ValueSignal

value = ValueSignal(prices).compute().loc[:LAST_COMPLETE_MONTH]
skewness = SkewnessSignal(daily_returns).compute().loc[:LAST_COMPLETE_MONTH]
final_signals["value"], final_signals["skewness"] = value, skewness

# 1. Coverage
coverage = pd.DataFrame(
    {
        "value_first": value.apply(lambda s: s.first_valid_index().strftime("%Y-%m")),
        "skew_first": skewness.apply(lambda s: s.first_valid_index().strftime("%Y-%m")),
    }
).join(UNIVERSE.to_frame()["sector"])
print("First valid month per contract")
print(coverage.T.to_string(), "\n")

# 2. Sanity check at the latest month
latest = LAST_COMPLETE_MONTH
for label, frame, high, low in [
    ("Value", value, "cheapest (long)", "most expensive (short)"),
    ("Skewness signal (-skew)", skewness, "most negative skew (long)", "most positive skew (short)"),
]:
    ranked = frame.loc[latest].dropna().sort_values(ascending=False).round(3)
    print(f"{label} at {latest:%Y-%m}")
    print(f"  {high:<27}: {ranked.head(5).to_dict()}")
    print(f"  {low:<27}: {ranked.tail(5).to_dict()}\n")

# 3. Rank IC by subperiod
ic_rows = {}
for name in ["value", "skewness"]:
    ic_all = rank_ic(final_signals[name].loc[BACKTEST_START:], forward_returns).dropna()
    for label, (start, end) in PERIODS.items():
        ic = ic_all.loc[start:end]
        if len(ic) < 24:
            continue
        ic_rows[(name, label)] = {
            "mean_ic": ic.mean(),
            "t_stat": ic.mean() / ic.std() * np.sqrt(len(ic)),
            "pct_positive": ic.gt(0).mean() * 100,
            "months": len(ic),
            "start": ic.index[0].strftime("%Y-%m"),
        }
print("Rank IC by subperiod (signal at t vs return in t+1)")
print(pd.DataFrame(ic_rows).T.to_string(), "\n")

# 4. Relationship with existing factors
print("Average monthly rank correlation")
for left, right in [("value", "mom_12_1"), ("value", "carry"), ("skewness", "mom_12_1"),
                    ("skewness", "carry"), ("value", "skewness")]:
    print(f"  {left:<9} vs {right:<9}: {mean_rank_corr(final_signals[left], final_signals[right]):.3f}")

First valid month per contract
                  CL       BZ       NG       HO       RB       GC       SI       HG       PL       PA       ZC       ZW       KE       ZS       ZM       ZL       ZO       ZR       KC       SB       CC       CT       OJ         LE         HE         GF
value_first  2005-11  2012-10  2005-11  2005-12  2006-02  2005-11  2005-11  2005-11  2015-04  2015-04  2005-10  2005-10  2005-12  2005-12  2005-08  2005-06  2009-04  2005-04  2005-04  2005-06  2005-04  2005-04  2006-12    2010-10    2006-03    2006-07
skew_first   2001-06  2008-06  2001-07  2001-07  2001-09  2001-07  2001-07  2001-07  2010-10  2010-11  2001-06  2001-06  2001-07  2001-07  2001-04  2002-01  2004-10  2000-10  2000-10  2000-12  2000-10  2000-10  2002-07    2006-04    2001-10    2002-01
sector        Energy   Energy   Energy   Energy   Energy   Metals   Metals   Metals   Metals   Metals   Grains   Grains   Grains   Grains   Grains   Grains   Grains   Grains    Softs    Softs    Softs    Softs    

In [12]:
%%writefile -a src/commodity_factors/config.py


# ----------------------------------------------------------- CFTC market codes
# CFTC Contract Market Codes (Legacy COT reports). Verified via the market-name
# audit file written by CFTCLegacySource (data/raw/cftc_legacy_markets.csv).
CFTC_MARKET_CODES: dict[str, str] = {
    "CL": "067651", "BZ": "06765T", "NG": "023651", "HO": "022651", "RB": "111659",
    "GC": "088691", "SI": "084691", "HG": "085692", "PL": "076651", "PA": "075651",
    "ZC": "002602", "ZW": "001602", "KE": "001612", "ZS": "005602", "ZM": "026603",
    "ZL": "007601", "ZO": "004603", "ZR": "039601",
    "KC": "083731", "SB": "080732", "CC": "073732", "CT": "033661", "OJ": "040701",
    "LE": "057642", "HE": "054642", "GF": "061641",
}

Appending to src/commodity_factors/config.py


In [13]:
%%writefile src/commodity_factors/data/cftc.py
"""Weekly Commitments of Traders positions (Legacy, futures only) from the CFTC."""

from __future__ import annotations

import io
import zipfile
from datetime import date
from pathlib import Path

import pandas as pd
import requests

from commodity_factors.data.base import CachedDataSource


class CFTCLegacySource(CachedDataSource):
    """Legacy futures-only COT positions for the mapped contracts.

    Downloads the 1986-2016 history file plus one file per year from 2017, keeps the
    mapped CFTC contract market codes and returns a weekly panel indexed by the report
    "as of" date (Tuesday), with (field, symbol) columns. Reports are published the
    following Friday; the release lag is applied when signals are built, not here.
    A market-name audit file is saved next to the cache so the mapping can be checked.
    """

    name = "cftc_legacy"
    BASE_URL = "https://www.cftc.gov/files/dea/history/"
    HISTORY_FILE = "deacot1986_2016.zip"
    YEAR_FILE = "deacot{year}.zip"
    FIRST_YEAR_FILE = 2017
    COLUMNS = {
        "Market and Exchange Names": "market_name",
        "As of Date in Form YYYY-MM-DD": "date",
        "CFTC Contract Market Code": "code",
        "Open Interest (All)": "open_interest",
        "Noncommercial Positions-Long (All)": "noncomm_long",
        "Noncommercial Positions-Short (All)": "noncomm_short",
        "Commercial Positions-Long (All)": "comm_long",
        "Commercial Positions-Short (All)": "comm_short",
    }

    def __init__(
        self,
        market_codes: dict[str, str],
        cache_dir: Path,
        last_year: int | None = None,
        timeout: float = 120.0,
    ) -> None:
        super().__init__(cache_dir)
        self._code_to_symbol = {code: symbol for symbol, code in market_codes.items()}
        self._last_year = last_year or date.today().year
        self._timeout = timeout

    @property
    def audit_path(self) -> Path:
        return self._cache_dir / f"{self.name}_markets.csv"

    def market_audit(self) -> pd.DataFrame:
        """Market names seen per contract code, with first and last report dates."""
        return pd.read_csv(self.audit_path, dtype={"code": str})

    def fetch(self) -> pd.DataFrame:
        files = [self.HISTORY_FILE] + [
            self.YEAR_FILE.format(year=year)
            for year in range(self.FIRST_YEAR_FILE, self._last_year + 1)
        ]
        raw = pd.concat([self._read_zip(file) for file in files], ignore_index=True)
        raw = raw[raw["code"].isin(self._code_to_symbol)].copy()
        raw["symbol"] = raw["code"].map(self._code_to_symbol)
        raw["date"] = pd.to_datetime(raw["date"])

        self._cache_dir.mkdir(parents=True, exist_ok=True)
        (
            raw.groupby(["symbol", "code", "market_name"])["date"]
            .agg(first="min", last="max", reports="count")
            .reset_index()
            .sort_values(["symbol", "first"])
            .to_csv(self.audit_path, index=False)
        )

        fields = [c for c in self.COLUMNS.values() if c not in {"market_name", "date", "code"}]
        panel = raw.pivot_table(index="date", columns="symbol", values=fields, aggfunc="last")
        panel.columns = panel.columns.set_names(["field", "symbol"])
        return panel.sort_index(axis=1)

    def _read_zip(self, file_name: str) -> pd.DataFrame:
        response = requests.get(self.BASE_URL + file_name, timeout=self._timeout)
        response.raise_for_status()
        with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
            member = next(n for n in archive.namelist() if n.lower().endswith((".txt", ".csv")))
            with archive.open(member) as handle:
                frame = pd.read_csv(handle, dtype=str, low_memory=False)

        frame.columns = frame.columns.str.strip()
        frame = frame[list(self.COLUMNS)].rename(columns=self.COLUMNS)
        frame["code"] = frame["code"].str.strip()
        frame["market_name"] = frame["market_name"].str.strip()
        numeric = [c for c in frame.columns if c not in {"market_name", "date", "code"}]
        frame[numeric] = frame[numeric].apply(
            lambda col: pd.to_numeric(col.str.replace(",", "").str.strip(), errors="coerce")
        )
        return frame

Writing src/commodity_factors/data/cftc.py


In [14]:
# ============================================================
# 03 · Cell 14 — Download COT data and audit the market-code mapping
# ============================================================
from commodity_factors.config import CFTC_MARKET_CODES
from commodity_factors.data.cftc import CFTCLegacySource

cftc = CFTCLegacySource(CFTC_MARKET_CODES, cache_dir=PATHS.raw)
cot = cftc.load(refresh=False)

print(f"COT panel : {cot.shape[0]:,} weekly reports | "
      f"{cot.index.min():%Y-%m-%d} -> {cot.index.max():%Y-%m-%d}")
print(f"Cached at : {cftc.cache_path}\n")

# 1. Market-name audit
audit = cftc.market_audit()
print("Market names matched to each code (check every name fits its contract)")
print(audit.to_string(index=False), "\n")

# 2. Missing contracts
found = set(cot.columns.get_level_values("symbol"))
missing = sorted(set(CFTC_MARKET_CODES) - found)
print(f"Contracts with no COT data: {missing if missing else 'none'}\n")

# 3. Coverage and hedger positioning
comm_net = (cot["comm_long"] - cot["comm_short"]) / cot["open_interest"] * 100
coverage = pd.DataFrame(
    {
        "first_report": cot["open_interest"].apply(lambda s: s.first_valid_index().strftime("%Y-%m-%d")),
        "last_report": cot["open_interest"].apply(lambda s: s.last_valid_index().strftime("%Y-%m-%d")),
        "reports": cot["open_interest"].notna().sum(),
        "comm_net_%oi_mean": comm_net.loc["2002":].mean().round(1),
        "pct_weeks_comm_net_short": comm_net.loc["2002":].lt(0).mean().mul(100).round(1),
    }
).reindex(UNIVERSE.symbols)
print("Coverage and commercial (hedger) net positioning since 2002")
print(coverage.to_string())

COT panel : 1,940 weekly reports | 1986-01-15 -> 2026-10-13
Cached at : C:\Users\navon\OneDrive\Desktop\Cmmodities Factor Modeling\data\raw\cftc_legacy.parquet

Market names matched to each code (check every name fits its contract)
symbol   code                                                           market_name      first       last  reports
    BZ 06765T               BRENT CRUDE OIL LAST DAY - NEW YORK MERCANTILE EXCHANGE 2011-10-18 2022-02-01      527
    BZ 06765T                         BRENT LAST DAY - NEW YORK MERCANTILE EXCHANGE 2022-02-08 2026-10-13      245
    CC 073732                                COCOA - COFFEE, SUGAR & COCOA EXCHANGE 1986-01-15 1999-12-14      537
    CC 073732                                COCOA - COFFEE,COCOA AND SUGAR EXCHANG 1999-12-21 2000-01-04        3
    CC 073732                                COCOA - COFFEE,SUGAR AND COCOA EXCHANG 2000-01-11 2002-04-23      119
    CC 073732                               COCOA - COFFEE,SUGAR AND COCOA EXC

In [15]:
%%writefile -a src/commodity_factors/signals.py


class HedgingPressureSignal(Signal):
    """Trader positioning from weekly COT reports, smoothed and lagged to its release.

    commercial : (short - long) / (short + long) of commercial traders. High = hedgers
                 net short, so longs earn the insurance premium (Basu & Miffre, 2013).
    speculator : (long - short) / (long + short) of non-commercial traders.

    Values are averaged over the trailing window of reports, then dated by when they
    could first be traded: the Tuesday position date plus ``release_lag_days``
    (reports are published Friday after most settlements; 7 days is conservative).
    """

    name = "hedging_pressure"

    def __init__(
        self,
        cot: pd.DataFrame,
        trader: str = "commercial",
        window_weeks: int = 52,
        min_weeks: int = 40,
        release_lag_days: int = 7,
    ) -> None:
        if trader not in {"commercial", "speculator"}:
            raise ValueError("trader must be 'commercial' or 'speculator'.")
        self._cot = cot
        self._trader = trader
        self._window_weeks = window_weeks
        self._min_weeks = min_weeks
        self._release_lag_days = release_lag_days

    def weekly(self) -> pd.DataFrame:
        """Unsmoothed weekly positioning, indexed by the report's position date."""
        if self._trader == "commercial":
            long, short = self._cot["comm_long"], self._cot["comm_short"]
            return (short - long) / (short + long)
        long, short = self._cot["noncomm_long"], self._cot["noncomm_short"]
        return (long - short) / (long + short)

    def compute(self) -> pd.DataFrame:
        smoothed = self.weekly().rolling(self._window_weeks, min_periods=self._min_weeks).mean()
        smoothed.index = smoothed.index + pd.Timedelta(days=self._release_lag_days)
        return month_end_last(smoothed)

Appending to src/commodity_factors/signals.py


In [16]:
# ============================================================
# 03 · Cell 16 — Hedging pressure: compute, timing check, preview
# ============================================================
from commodity_factors.signals import HedgingPressureSignal

hp_model = HedgingPressureSignal(cot, trader="commercial")
hp_comm = hp_model.compute().loc[:LAST_COMPLETE_MONTH]
hp_spec = HedgingPressureSignal(cot, trader="speculator").compute().loc[:LAST_COMPLETE_MONTH]
final_signals["hedging_pressure"], final_signals["hp_speculator"] = hp_comm, hp_spec

# 1. Timing check: which report feeds the CL signal at 2024-05-31 (a Friday)?
check_month = pd.Timestamp("2024-05-31")
smoothed = hp_model.weekly()["CL"].rolling(52, min_periods=40).mean()
usable = smoothed.loc[: check_month - pd.Timedelta(days=7)]
print("Timing check: CL hedging pressure at 2024-05-31")
print(f"  report used (position date): {usable.index[-1]:%Y-%m-%d} (expected 2024-05-21)")
print(f"  manual value : {usable.iloc[-1]:.6f}")
print(f"  signal panel : {hp_comm.at[check_month, 'CL']:.6f}\n")

# 2. Coverage and latest ranking
first = hp_comm.apply(lambda s: s.first_valid_index().strftime("%Y-%m"))
print("First valid month (commercial):", ", ".join(f"{s} {d}" for s, d in first.items()), "\n")
latest = hp_comm.loc[LAST_COMPLETE_MONTH].dropna().sort_values(ascending=False).round(3)
print(f"Hedging pressure at {LAST_COMPLETE_MONTH:%Y-%m}")
print(f"  hedgers most net short (long) : {latest.head(5).to_dict()}")
print(f"  hedgers most net long (short) : {latest.tail(5).to_dict()}\n")

# 3. Rank IC by subperiod
ic_rows = {}
for name in ["hedging_pressure", "hp_speculator"]:
    ic_all = rank_ic(final_signals[name].loc[BACKTEST_START:], forward_returns).dropna()
    for label, (start, end) in PERIODS.items():
        ic = ic_all.loc[start:end]
        ic_rows[(name, label)] = {
            "mean_ic": round(ic.mean(), 4),
            "t_stat": round(ic.mean() / ic.std() * np.sqrt(len(ic)), 2),
            "pct_positive": round(ic.gt(0).mean() * 100, 1),
            "months": len(ic),
        }
print("Rank IC by subperiod (signal at t vs return in t+1)")
print(pd.DataFrame(ic_rows).T.to_string(), "\n")

# 4. Correlations
print("Average monthly rank correlation")
for left, right in [("hedging_pressure", "hp_speculator"), ("hedging_pressure", "carry"),
                    ("hedging_pressure", "mom_12_1"), ("hedging_pressure", "value"),
                    ("hedging_pressure", "skewness"), ("hp_speculator", "mom_12_1")]:
    print(f"  {left:<16} vs {right:<14}: {mean_rank_corr(final_signals[left], final_signals[right]):.3f}")

Timing check: CL hedging pressure at 2024-05-31
  report used (position date): 2024-05-21 (expected 2024-05-21)
  manual value : 0.168838
  signal panel : 0.168838

First valid month (commercial): BZ 2012-10, CC 1987-09, CL 1987-09, CT 1987-09, GC 1987-09, GF 1987-09, HE 1997-02, HG 1991-03, HO 1987-09, KC 1987-09, KE 1998-10, LE 1987-09, NG 1991-12, OJ 1987-09, PA 1987-09, PL 1987-09, RB 2006-11, SB 1987-09, SI 1987-09, ZC 1998-10, ZL 1987-09, ZM 1987-09, ZO 1998-10, ZR 1995-07, ZS 1998-10, ZW 1998-10 

Hedging pressure at 2026-09
  hedgers most net short (long) : {'GC': 0.612, 'SI': 0.395, 'PL': 0.372, 'HG': 0.298, 'LE': 0.242}
  hedgers most net long (short) : {'SB': -0.123, 'ZW': -0.126, 'BZ': -0.156, 'NG': -0.162, 'ZR': -0.178}

Rank IC by subperiod (signal at t vs return in t+1)
                            mean_ic  t_stat  pct_positive  months
hedging_pressure 2002-2026   0.0427    3.07          56.4   296.0
                 2002-2008   0.0512    1.76          52.4    84.0
      

In [17]:
%%writefile -a src/commodity_factors/signals.py


class BasisMomentumSignal(Signal):
    """12-month return of the nearby contract minus that of the next contract.

    Captures changes in the front of the curve that carry and momentum miss
    (Boons & Prado, 2019). Both legs come from a stitched curve (contract 1 and 2);
    roll days are masked in both legs, and any day where either leg has a
    non-positive price is dropped from both so the difference stays unbiased.
    """

    name = "basis_momentum"

    def __init__(
        self,
        curves: pd.DataFrame,
        roll_dates: dict[str, pd.DatetimeIndex],
        window: int = 12,
        near: int = 1,
        far: int = 2,
    ) -> None:
        self._curves = curves
        self._roll_dates = roll_dates
        self._window = window
        self._near = near
        self._far = far

    def _symbol(self, symbol: str) -> pd.Series:
        pair = pd.DataFrame(
            {"near": self._curves[(symbol, self._near)], "far": self._curves[(symbol, self._far)]}
        ).dropna()
        log_ret = np.log(pair.where(pair > 0)).diff()
        log_ret.loc[log_ret.index.isin(self._roll_dates.get(symbol, pd.DatetimeIndex([])))] = np.nan
        log_ret = log_ret.where(log_ret.notna().all(axis=1))

        monthly = log_ret.resample("ME").sum(min_count=1)
        has_data = monthly.notna().all(axis=1)
        trailing = monthly.fillna(0.0).rolling(self._window).sum()
        complete = has_data.astype(int).rolling(self._window).sum().eq(self._window)
        return (trailing["near"] - trailing["far"]).where(complete).rename(symbol)

    def compute(self) -> pd.DataFrame:
        symbols = self._curves.columns.get_level_values("symbol").unique()
        return pd.DataFrame({symbol: self._symbol(symbol) for symbol in symbols})

Appending to src/commodity_factors/signals.py


In [18]:
# ============================================================
# 03 · Cell 18 — Basis-momentum, demeaned HP, correlation matrix, save
# ============================================================
from commodity_factors.signals import BasisMomentumSignal, drop_incomplete_last_month

# 1. Basis-momentum (energy, EIA curves)
energy = curves.columns.get_level_values("symbol").unique().tolist()
eia_roll_dates = {
    s: roll_calendar.roll_dates(s, curves[(s, 1)].dropna().loc["2000":].index) for s in energy
}
basis_mom = BasisMomentumSignal(curves, eia_roll_dates).compute()
basis_mom = drop_incomplete_last_month(basis_mom, curves.dropna(how="all").index.max())
final_signals["basis_momentum"] = basis_mom

print(f"Basis-momentum: {basis_mom.dropna(how='all').index.min():%Y-%m} -> "
      f"{basis_mom.dropna(how='all').index.max():%Y-%m}\n")
print("Time-series preview: BASIS-MOMENTUM (long if > 0, short if < 0)")
print(time_series_preview(basis_mom, forward_returns).to_string(), "\n")

overlap = {
    s: {
        "corr_with_eia_carry": basis_mom[s].corr(carry_raw[s]),
        "corr_with_roll_yield": basis_mom[s].corr(carry[s]),
    }
    for s in energy
}
print("Time-series correlation of basis-momentum with carry measures")
print(pd.DataFrame(overlap).T.round(3).to_string(), "\n")

# 2. Hedging pressure: levels vs demeaned (own history, strictly past)
hp_demeaned = hp_comm - hp_comm.expanding(min_periods=36).mean().shift(1)
final_signals["hp_demeaned"] = hp_demeaned

ic_rows = {}
for name, frame in [("hp_levels", hp_comm), ("hp_demeaned", hp_demeaned)]:
    ic_all = rank_ic(frame.loc[BACKTEST_START:], forward_returns).dropna()
    for label, (start, end) in PERIODS.items():
        ic = ic_all.loc[start:end]
        ic_rows[(name, label)] = {
            "mean_ic": round(ic.mean(), 4),
            "t_stat": round(ic.mean() / ic.std() * np.sqrt(len(ic)), 2),
            "months": len(ic),
        }
print("Hedging pressure: levels vs demeaned")
print(pd.DataFrame(ic_rows).T.to_string(), "\n")

precious = ["GC", "SI", "PL", "PA"]
for name, frame in [("levels", hp_comm), ("demeaned", hp_demeaned)]:
    sample = frame.loc[BACKTEST_START:]
    in_top_third = sample.rank(axis=1, pct=True).gt(2 / 3)
    share = in_top_third[precious].sum(axis=1).div(in_top_third.sum(axis=1)).mean() * 100
    print(f"  Precious metals' average share of the top third ({name}): {share:.1f}%")
print()

# 3. Correlation matrix and IC summary of cross-sectional factors
cross_sectional = ["mom_12_1", "carry", "value", "skewness", "hedging_pressure", "hp_demeaned"]
corr_matrix = pd.DataFrame(
    {a: {b: mean_rank_corr(final_signals[a], final_signals[b]) for b in cross_sectional}
     for a in cross_sectional}
)
print("Average monthly rank correlation between cross-sectional factors")
print(corr_matrix.round(2).to_string(), "\n")

ic_summary = {}
for name in cross_sectional:
    ic = rank_ic(final_signals[name].loc[BACKTEST_START:], forward_returns).dropna()
    ic_summary[name] = {
        "start": ic.index[0].strftime("%Y-%m"),
        "mean_ic": round(ic.mean(), 4),
        "t_stat": round(ic.mean() / ic.std() * np.sqrt(len(ic)), 2),
        "pct_positive": round(ic.gt(0).mean() * 100, 1),
    }
print("IC summary: all cross-sectional factors (full sample)")
print(pd.DataFrame(ic_summary).T.to_string(), "\n")

# 4. Save
signal_panel = pd.concat(final_signals, axis=1, names=["signal", "symbol"], sort=True)
signal_panel.to_parquet(PATHS.processed / "signals.parquet")
saved = pd.DataFrame(
    {
        name: {
            "first": frame.dropna(how="all").index.min().strftime("%Y-%m"),
            "last": frame.dropna(how="all").index.max().strftime("%Y-%m"),
            "contracts": frame.notna().any().sum(),
        }
        for name, frame in final_signals.items()
    }
).T
print("Saved signals.parquet")
print(saved.to_string())

Basis-momentum: 1985-12 -> 2024-03

Time-series preview: BASIS-MOMENTUM (long if > 0, short if < 0)
    months  pct_long  ret_long_%  ret_short_%  position_ret_%  t_stat
CL   267.0     46.82        1.27         0.49            0.33    0.48
HO   267.0     56.93        0.85         1.50           -0.16   -0.27
NG   267.0     66.67       -0.68         0.43           -0.60   -0.61
RB   211.0     73.46        0.57         5.10           -0.93   -1.27 

Time-series correlation of basis-momentum with carry measures
    corr_with_eia_carry  corr_with_roll_yield
CL                0.305                 0.011
HO                0.444                 0.563
NG                0.175                 0.409
RB                0.258                 0.327 

Hedging pressure: levels vs demeaned
                       mean_ic  t_stat  months
hp_levels   2002-2026   0.0427    3.07   296.0
            2002-2008   0.0512    1.76    84.0
            2009-2026   0.0393    2.51   212.0
hp_demeaned 2002-2026   0.042

In [19]:
# ============================================================
# 03 · Cell 19 — Trim pre-2001 basis-momentum (unmasked rolls) and re-save
# ============================================================
final_signals["basis_momentum"] = basis_mom.loc["2000-12-31":]

signal_panel = pd.concat(final_signals, axis=1, names=["signal", "symbol"], sort=True)
signal_panel.to_parquet(PATHS.processed / "signals.parquet")

bm = final_signals["basis_momentum"].dropna(how="all")
print(f"basis_momentum now: {bm.index.min():%Y-%m} -> {bm.index.max():%Y-%m}")
print(f"Saved signals.parquet: {len(final_signals)} signals, "
      f"{signal_panel.shape[0]} months x {signal_panel.shape[1]} columns")

basis_momentum now: 2000-12 -> 2024-03
Saved signals.parquet: 12 signals, 501 months x 246 columns
